# 📦 ASER Dataset — Preparation
**Project:** Indian Children Speech Recognition (ICASSP)
**What this notebook does:**
- Unzips all 5,301 ASER child session archives
- Reads each child's JSON metadata file
- Builds a single master `manifest.csv` mapping every audio file to its transcript + metadata

Run cells **top to bottom**, once. After this notebook finishes, move to `02_dataset_eda_and_splits.ipynb`.


## Cell 1 — Imports and Path Setup
We define where the raw ASER zips live, where to extract them, and where to save the manifest.


In [ ]:
import os
import json
import zipfile
import csv
from pathlib import Path
from collections import defaultdict

# ── Paths ──────────────────────────────────────────────────────────────────
ASER_ROOT   = Path("/home/hp/Indain_children_spech/ASER-Dataset/Data")
EXTRACT_DIR = Path("/home/hp/Indain_children_spech/ASER-Dataset/extracted")
MANIFEST    = Path("/home/hp/Indain_children_spech/ASER-Dataset/manifest.csv")

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Raw data location : {ASER_ROOT}")
print(f"Extract to        : {EXTRACT_DIR}")
print(f"Manifest output   : {MANIFEST}")


## Cell 2 — Language and Level Mappings
`que_id` encodes language and reading level in its name (e.g. `HI_S1_ST_0`).
We decode these into human-readable labels here.


In [ ]:
# Maps folder name → broad language
FOLDER_LANG = {
    "Hindi RJ":   "Hindi",
    "Hindi UP":   "Hindi",
    "Marathi MH": "Marathi",
}

# Maps level code → full reading level name
LEVEL_MAP = {
    "ST": "Story",
    "P":  "Paragraph",
    "WD": "Word",
    "L":  "Letter",
    "CL": "Capital Letter (English)",
    "SL": "Small Letter (English)",
    "W":  "Word (English)",
    "S":  "Sentence (English)",
}

def get_level_and_lang(que_id):
    """
    Input : 'HI_S1_ST_0'
    Output: ('Story', 'Hindi')
    """
    parts = que_id.split("_")
    if len(parts) < 3:
        return "Unknown", "Unknown"
    lang_code  = parts[0]   # HI or MR
    level_code = parts[2]   # ST, P, WD, L, CL, SL, W, S

    script_lang = "Hindi" if lang_code == "HI" else "Marathi" if lang_code == "MR" else "Unknown"
    if level_code in ("CL", "SL", "W", "S"):
        script_lang = "English"   # English-task items inside any session

    return LEVEL_MAP.get(level_code, level_code), script_lang

# Quick test
print(get_level_and_lang("HI_S1_ST_0"))   # Expected: ('Story', 'Hindi')
print(get_level_and_lang("MR_S2_CL_1"))   # Expected: ('Capital Letter (English)', 'English')


## Cell 3 — Extract All Zips + Build Manifest
This is the main loop. For each of 5,301 zip files:
1. Extract the .mp3 audio files to disk
2. Read the summary JSON for transcript and metadata
3. Add one row per audio clip to our records list


In [ ]:
records = []
errors  = []

zip_files = sorted(ASER_ROOT.rglob("*.zip"))
total     = len(zip_files)
print(f"Found {total} zip files. Starting extraction...")

for i, zip_path in enumerate(zip_files, 1):
    if i % 500 == 0 or i == 1:
        print(f"  Processing {i}/{total} ...")

    folder_name = zip_path.parent.name   # e.g. "Hindi RJ"
    child_id    = zip_path.stem          # e.g. "3439"
    child_out   = EXTRACT_DIR / folder_name / child_id
    child_out.mkdir(parents=True, exist_ok=True)

    # --- Step 1: Extract zip ---
    try:
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(child_out)
    except Exception as e:
        errors.append((str(zip_path), str(e)))
        continue

    # --- Step 2: Find and parse the JSON ---
    json_files = list(child_out.glob("summary*.json"))
    if not json_files:
        errors.append((str(zip_path), "No summary JSON found"))
        continue

    with open(json_files[0], "r", encoding="utf-8") as f:
        try:
            meta = json.load(f)
        except Exception as e:
            errors.append((str(zip_path), f"JSON parse error: {e}"))
            continue

    age_group           = meta.get("ageGroup", "Unknown")
    stud_class          = meta.get("studClass", "Unknown")
    native_proficiency  = meta.get("nativeProficiency", "Unknown")
    english_proficiency = meta.get("englishProficiency", "Unknown")

    # --- Step 3: One row per audio clip ---
    for item in meta.get("sequenceList", []):
        que_text   = item.get("que_text", "").strip()
        rec_name   = item.get("recordingName", "")
        que_id     = item.get("que_id", "")
        is_correct = item.get("isCorrect", None)
        n_mistakes = item.get("noOfMistakes", "0")

        audio_path = child_out / rec_name
        if not audio_path.exists():
            errors.append((str(zip_path), f"Missing audio: {rec_name}"))
            continue

        reading_level, script_lang = get_level_and_lang(que_id)

        records.append({
            "audio_path":          str(audio_path),
            "transcript":          que_text,
            "script_language":     script_lang,
            "region":              folder_name,
            "reading_level":       reading_level,
            "que_id":              que_id,
            "is_correct":          is_correct,
            "num_mistakes":        n_mistakes,
            "age_group":           age_group,
            "student_class":       stud_class,
            "native_proficiency":  native_proficiency,
            "english_proficiency": english_proficiency,
            "child_id":            child_id,
        })

print(f"\nDone! Total records : {len(records)}")
print(f"Errors encountered  : {len(errors)}")
if errors:
    print("First 5 errors:")
    for e in errors[:5]:
        print("  ", e)


## Cell 4 — Save Manifest CSV
We save all records to a single CSV file.
Every ASR training framework reads data from a file like this.


In [ ]:
fieldnames = list(records[0].keys())

with open(MANIFEST, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(records)

print(f"Manifest saved → {MANIFEST}")
print(f"Total rows     : {len(records):,}")

# Preview first 3 rows
print("\nSample rows:")
for r in records[:3]:
    print(f"  audio : {Path(r['audio_path']).name}")
    print(f"  text  : {r['transcript'][:60]}")
    print(f"  lang  : {r['script_language']}  |  level: {r['reading_level']}  |  age: {r['age_group']}")
    print()
